# 6.21 — Ortogonalità

**Prerequisiti:** 6.20 Prodotti scalari, 6.8 Basi.


## Perché ci interessa

Nell'esercizio svolto del notebook 6.20 abbiamo trovato, "a mano", un vettore ortogonale a $1$ sottraendo la sua proiezione. Una **base ortogonale** — dove ogni coppia di vettori distinti è ortogonale — è particolarmente comoda: come vedremo, calcolare le coordinate di un vettore rispetto a una base ortogonale è immediato, senza risolvere sistemi lineari (a differenza di quanto fatto nel notebook 6.8). In questo notebook sviluppiamo un algoritmo — **Gram-Schmidt** — che trasforma sistematicamente una base qualsiasi in una base ortogonale.

## Teoria, passo dopo passo

### Insiemi ortogonali

Un insieme di vettori $\{v_1,\dots,v_k\}$ (tutti non nulli) è **ortogonale** se $\langle v_i,v_j\rangle=0$ per ogni $i\ne j$ (notebook 6.2, 6.20).

### Un insieme ortogonale è sempre linearmente indipendente

> **Fatto.** Un insieme ortogonale di vettori non nulli è linearmente indipendente (notebook 6.6).

**Dimostrazione.** Supponiamo $\lambda_1v_1+\cdots+\lambda_kv_k=\mathbf 0$. Calcoliamo il prodotto scalare di entrambi i membri con $v_i$ (per un indice $i$ fissato): $\langle\lambda_1v_1+\cdots+\lambda_kv_k,\,v_i\rangle=\langle\mathbf 0,v_i\rangle=0$. Per bilinearità, il primo membro si sviluppa in $\sum_j\lambda_j\langle v_j,v_i\rangle$; ma per ortogonalità, $\langle v_j,v_i\rangle=0$ per ogni $j\ne i$, quindi resta solo il termine $\lambda_i\langle v_i,v_i\rangle=0$. Poiché $v_i\ne\mathbf 0$, per la positività $\langle v_i,v_i\rangle\ne0$ (notebook 6.20), quindi $\lambda_i=0$. Questo vale per ogni $i$: tutti i coefficienti sono nulli. $\blacksquare$

### Coordinate immediate rispetto a una base ortogonale

Se $\mathcal B=\{v_1,\dots,v_n\}$ è una base **ortogonale** di $V$, le coordinate di un vettore $u$ si calcolano senza risolvere alcun sistema: $u=\lambda_1v_1+\cdots+\lambda_nv_n$ con

$$\lambda_i = \frac{\langle u,v_i\rangle}{\langle v_i,v_i\rangle}$$

**Perché:** calcolando $\langle u,v_i\rangle$ e usando la bilinearità come nella dimostrazione sopra, tutti i termini con $j\ne i$ si annullano per ortogonalità, lasciando $\langle u,v_i\rangle=\lambda_i\langle v_i,v_i\rangle$: risolvendo per $\lambda_i$ (divisione lecita, $\langle v_i,v_i\rangle\ne0$) si ottiene la formula. Riconosciamo che $\lambda_iv_i$ è proprio la **proiezione** di $u$ su $v_i$ (notebook 6.2): un vettore, in base ortogonale, si decompone semplicemente sommando le sue proiezioni su ciascun vettore di base.

### Il procedimento di Gram-Schmidt

Data una base qualsiasi $\{w_1,\dots,w_n\}$ di $V$, costruiamo una base ortogonale $\{v_1,\dots,v_n\}$ dello stesso spazio, uno alla volta:

$$v_1 = w_1$$
$$v_2 = w_2 - \frac{\langle w_2,v_1\rangle}{\langle v_1,v_1\rangle}v_1$$
$$v_3 = w_3 - \frac{\langle w_3,v_1\rangle}{\langle v_1,v_1\rangle}v_1 - \frac{\langle w_3,v_2\rangle}{\langle v_2,v_2\rangle}v_2$$

e così via: a ogni passo, si sottrae da $w_k$ la sua proiezione su **tutti** i vettori ortogonali già costruiti $v_1,\dots,v_{k-1}$, ottenendo un vettore $v_k$ automaticamente ortogonale a tutti loro — è la stessa idea usata nell'esercizio svolto del notebook 6.20, ripetuta sistematicamente.

In [ ]:
import sympy as sp

def proiezione(w, v):
    return (w.dot(v) / v.dot(v)) * v

# Base di partenza in R^3 (non ortogonale)
w1 = sp.Matrix([1, 1, 0])
w2 = sp.Matrix([1, 0, 1])
w3 = sp.Matrix([0, 1, 1])

# Gram-Schmidt passo per passo
v1 = w1
v2 = w2 - proiezione(w2, v1)
v3 = w3 - proiezione(w3, v1) - proiezione(w3, v2)

print("v1 =", v1.T)
print("v2 =", v2.T)
print("v3 =", v3.T)

# Verifica ortogonalità
print("\nv1.v2 =", v1.dot(v2))
print("v1.v3 =", v1.dot(v3))
print("v2.v3 =", v2.dot(v3))

# Confronto con GramSchmidt di SymPy
base_ortogonale = sp.GramSchmidt([w1, w2, w3])
print("\nBase ortogonale (SymPy):", [v.T for v in base_ortogonale])

## Esercizio svolto e commentato

**Problema:** applica Gram-Schmidt a $w_1=(1,1)$, $w_2=(0,1)$ (base di $\mathbb R^2$), poi usa la base ortogonale ottenuta per trovare le coordinate di $u=(3,1)$.

**Ragionamento:**

**Passo 1 — poniamo $v_1=w_1=(1,1)$** (il primo vettore resta invariato, per definizione di Gram-Schmidt).

**Passo 2 — calcoliamo la proiezione di $w_2$ su $v_1$**: $\operatorname{proj}_{v_1}w_2=\dfrac{\langle w_2,v_1\rangle}{\langle v_1,v_1\rangle}v_1=\dfrac{0\times1+1\times1}{1^2+1^2}(1,1)=\dfrac12(1,1)=(0.5,0.5)$.

**Passo 3 — sottraiamo per ottenere $v_2$**: $v_2=w_2-\operatorname{proj}_{v_1}w_2=(0,1)-(0.5,0.5)=(-0.5,0.5)$.

**Passo 4 — verifichiamo l'ortogonalità**: $\langle v_1,v_2\rangle=1\times(-0.5)+1\times0.5=-0.5+0.5=0$. ✓

**Passo 5 — usiamo la formula delle coordinate immediate** per $u=(3,1)$ rispetto a $\{v_1,v_2\}$: $\lambda_1=\dfrac{\langle u,v_1\rangle}{\langle v_1,v_1\rangle}=\dfrac{3\times1+1\times1}{2}=\dfrac42=2$. $\lambda_2=\dfrac{\langle u,v_2\rangle}{\langle v_2,v_2\rangle}=\dfrac{3\times(-0.5)+1\times0.5}{0.25+0.25}=\dfrac{-1}{0.5}=-2$.

**Verifica**: $2v_1-2v_2=2(1,1)-2(-0.5,0.5)=(2,2)-(-1,1)=(3,1)=u$. ✓

**Perché questo metodo evita completamente di risolvere un sistema lineare** (a differenza di quanto fatto nell'esercizio del notebook 6.8, dove per una base non ortogonale $\{(1,1),(1,-1)\}$ avevamo comunque impostato e risolto un sistema $2\times2$): il "costo" di Gram-Schmidt è pagato **una volta sola**, nel costruire la base ortogonale; da lì in poi, le coordinate di **ogni** vettore si trovano con un semplice calcolo di prodotti scalari (formula diretta), senza mai dover risolvere un sistema — un vantaggio enorme se dobbiamo trovare le coordinate di molti vettori diversi rispetto alla stessa base.

In [ ]:
import sympy as sp

v1 = sp.Matrix([1, 1])
v2 = sp.Matrix([sp.Rational(-1,2), sp.Rational(1,2)])
u = sp.Matrix([3, 1])

lambda1 = u.dot(v1) / v1.dot(v1)
lambda2 = u.dot(v2) / v2.dot(v2)
print("lambda1 =", lambda1, " | lambda2 =", lambda2)

verifica = lambda1 * v1 + lambda2 * v2
print("Verifica lambda1*v1 + lambda2*v2 =", verifica.T)

## Esercizi da risolvere

1. Applica Gram-Schmidt a $w_1=(2,0)$, $w_2=(1,3)$ in $\mathbb R^2$.
2. Verifica che l'insieme $\{(1,0,0),(0,1,1),(0,1,-1)\}$ è già ortogonale (senza bisogno di Gram-Schmidt).
3. Usando la base ortogonale dell'esercizio 2, trova le coordinate di $u=(2,4,0)$ (con la formula diretta, non un sistema).
4. *(più impegnativo)* Una base **ortonormale** è una base ortogonale i cui vettori hanno tutti norma $1$. Normalizza la base ortogonale trovata nell'esercizio 1 (dividi ogni vettore per la sua norma) per ottenere una base ortonormale.

<details>
<summary><strong>👉 Soluzioni (clicca per visualizzare)</strong></summary>

**1.** $v_1=(2,0)$. $\operatorname{proj}_{v_1}w_2=\dfrac{1\times2+3\times0}{4}(2,0)=\dfrac24(2,0)=(1,0)$. $v_2=(1,3)-(1,0)=(0,3)$. Verifica: $v_1\cdot v_2=2\times0+0\times3=0$. ✓

**2.** $(1,0,0)\cdot(0,1,1)=0$. $(1,0,0)\cdot(0,1,-1)=0$. $(0,1,1)\cdot(0,1,-1)=0+1-1=0$. Tutti i prodotti scalari sono nulli: **ortogonale**.

**3.** $\lambda_1=\dfrac{u\cdot(1,0,0)}{\|(1,0,0)\|^2}=\dfrac{2}{1}=2$. $\lambda_2=\dfrac{u\cdot(0,1,1)}{\|(0,1,1)\|^2}=\dfrac{0+4+0}{2}=2$. $\lambda_3=\dfrac{u\cdot(0,1,-1)}{\|(0,1,-1)\|^2}=\dfrac{0+4-0}{2}=2$. Verifica: $2(1,0,0)+2(0,1,1)+2(0,1,-1)=(2,0,0)+(0,2,2)+(0,2,-2)=(2,4,0)=u$. ✓

**4.** $\|v_1\|=\|(2,0)\|=2$, quindi $\hat v_1=(1,0)$. $\|v_2\|=\|(0,3)\|=3$, quindi $\hat v_2=(0,1)$. Base ortonormale: $\{(1,0),(0,1)\}$ (in questo caso particolare, coincide con la base canonica).

Verifica computazionale:
```python
import sympy as sp

w1, w2 = sp.Matrix([2,0]), sp.Matrix([1,3])
proj = (w2.dot(w1)/w1.dot(w1)) * w1
v2 = w2 - proj
print("Es.1:", v2.T)

a, b, c = sp.Matrix([1,0,0]), sp.Matrix([0,1,1]), sp.Matrix([0,1,-1])
print("Es.2:", a.dot(b), a.dot(c), b.dot(c))
```

</details>

## Riepilogo finale

- Un insieme **ortogonale** di vettori non nulli è sempre linearmente indipendente.
- In una base ortogonale, le coordinate si calcolano con una formula diretta $\lambda_i=\langle u,v_i\rangle/\langle v_i,v_i\rangle$, senza risolvere sistemi lineari — la somma delle proiezioni di $u$ su ciascun vettore di base.
- Il procedimento di **Gram-Schmidt** trasforma una base qualsiasi in una base ortogonale, sottraendo a ogni vettore le sue proiezioni sui precedenti già ortogonalizzati.
- Una base **ortonormale** (ortogonale e con vettori di norma 1) è il caso più comodo di tutti; la ricaviamo semplicemente normalizzando una base ortogonale.

## Collegamenti

Il prossimo notebook è **6.22 — Basi ortonormali**: approfondiamo le basi ortonormali, viste appena qui, e le loro proprietà speciali per il calcolo delle coordinate e delle matrici associate.